# CriticalCare Copilot: Master Pipeline

This unified kernel handles the complete E2E clinical LLM lifecycle:
1. **Bootstrap**: Link GitHub repo and install medical-specialized libraries.
2. **Data Acquisition**: Securely download restricted MIMIC-IV and eICU tables from PhysioNet.
3. **Preprocessing**: Build structured JSONL datasets using the clinical extraction harness.
4. **Training**: Execute QLoRA Supervised Fine-Tuning (SFT) on Gemma 4 E4B.
5. **Validation**: Run the automated benchmark harness with regression checks.
6. **Deployment**: Merge weights and push to the HuggingFace Inference Registry.

### Step 1: Bootstrap & Environment setup
We use Kaggle's internet gateway to fetch the latest hardened scripts and libraries.

In [ ]:
!pip -q install -U huggingface_hub transformers peft accelerate trl bitsandbytes datasets sentencepiece pandas yaml
!git clone https://github.com/hssling/criticalcare-copilot.git
import os
os.chdir('/kaggle/working/criticalcare-copilot')

### Step 2: PhysioNet Authorization & Data Ingestion

**Configuration:** Requires `PHYSIONET_USER` and `PHYSIONET_PASS` in **Add-ons -> Secrets**.

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

try:
    pn_user = user_secrets.get_secret('PHYSIONET_USER')
    pn_pass = user_secrets.get_secret('PHYSIONET_PASS')
except:
    pn_user, pn_pass = None, None

if pn_user and pn_pass:
    print('Downloading MIMIC-IV and eICU core tables...')
    !mkdir -p data/raw/mimiciv/hosp data/raw/mimiciv/icu data/raw/eicu
    # MIMIC-IV Core
    for f in ['hosp/patients.csv.gz', 'hosp/admissions.csv.gz', 'icu/icustays.csv.gz']:
        !wget -q --user {pn_user} --password {pn_pass} -O data/raw/mimiciv/{f} https://physionet.org/content/mimiciv/2.2/{f}
    # eICU Core
    !wget -q --user {pn_user} --password {pn_pass} -O data/raw/eicu/patient.csv.gz https://physionet.org/content/eicu-crd/2.0/patient.csv.gz
    
    os.environ['MIMIC_IV_ROOT'] = os.path.abspath('data/raw/mimiciv')
    os.environ['EICU_ROOT'] = os.path.abspath('data/raw/eicu')
else:
    print('WARNING: PhysioNet credentials missing. Falling back to synthetic samples.')

### Step 3: Dataset Generation
Constructing the instruction-tuning pairs.

In [ ]:
print('Building training/validation shards...')
!python -m data.builders.build_mimic_cases --out data/processed/train.jsonl --limit 5000
!python -m data.builders.build_eicu_cases   --out data/processed/valid.jsonl --limit 500
!python -m data.builders.build_n2c2_ade_cases --out data/processed/ade_cases.jsonl
!python -m data.builders.split_train_valid_test --inputs data/processed/*.jsonl --out data/processed

### Step 4: Model Training (QLoRA)
Specialized fine-tuning on the Tesla T4 backend.

In [ ]:
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'
print('Starting SFT Session...')
!python training/scripts/train_sft.py --config training/configs/gemma4_e4b_qlora.yaml

### Step 5: Safety Benchmarking
Verifying the newly trained adapter against safety thresholds.

In [ ]:
print('Running Automated Evaluation...')
!python eval/benchmark_runner.py --suite smoke --adapter /kaggle/working/checkpoints/gemma4_e4b_qlora

### Step 6: Registration & Hub Push
Deploying the final artifact to HuggingFace.

In [ ]:
try:
    os.environ['HF_HUB_TOKEN'] = user_secrets.get_secret('HF_HUB_TOKEN')
except:
    pass

if os.environ.get('HF_HUB_TOKEN'):
    print('Exporting and pushing to HuggingFace Registry...')
    os.environ['HF_ORG'] = 'hssling'
    os.environ['HF_MODEL_REPO'] = 'criticalcare-copilot'
    
    !python training/scripts/merge_lora.py --base google/gemma-4-e4b-it --adapter /kaggle/working/checkpoints/gemma4_e4b_qlora --out /kaggle/working/checkpoints/merged
    !python hf/publish_model.py --path /kaggle/working/checkpoints/merged --repo $HF_ORG/$HF_MODEL_REPO
else:
    print('ERROR: Set HF_HUB_TOKEN secret to enable automated deployment.')